# 제6장. AI가 지어낸 책을 믿어도 될까? — AI와 함께 읽기

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITPainter/korea-china-classics-text-mining/blob/main/notebooks/ch6_llm.ipynb)

4장에서 컴퓨터가 『장자』에서 찾아낸 ‘나비 꿈’ 구절을 이번에는 챗봇(AI)에게 물어보고, 그 답을 4장의 방법으로 검증합니다. 이어서 다국어 임베딩으로 원문과 번역문을 정렬하고, AI 활용 기록을 남깁니다. API 키는 필요 없습니다.

In [1]:
# 실습 환경 준비 (Colab에서 처음 한 번 실행)
import sys
if "google.colab" in sys.modules:
    !pip -q install "kclit @ git+https://github.com/ITPainter/korea-china-classics-text-mining.git" "sentence-transformers"
from kclit.viz import setup_korean_font
print("그래프 글꼴:", setup_korean_font())

그래프 글꼴: AppleGothic


## 6.1 번역 초안과 출전 묻기

『구운몽』 16회에서 육관대사가 『장자』를 들려주는 대목입니다. 아래 코드는 AI에게 보낼 질문 문장을 만들어 줍니다(AI를 직접 부르지는 않습니다). 출력된 문장을 Claude, ChatGPT, Gemini 같은 챗봇의 대화창에 붙여 넣어 답을 받으세요. 확신이 없는 곳은 [?]로 표시하고 출전을 지어내지 말라는 지시가 들어 있습니다.

In [2]:
from kclit import llm
passage = ("莊周夢爲蝴蝶，蝴蝶又變爲莊周。\n莊周之夢爲蝴蝶耶？\n"
           "蝴蝶之夢爲莊周耶？\n終不能辨之！")
print(llm.translate_prompt(passage))
print()
print(llm.allusion_prompt(passage))

당신은 한문과 고전 중국어를 한국어로 옮기는 번역 보조자입니다. 원문의 구절 순서를 지켜 직역에 가깝게 옮기고, 확신이 없는 글자나 전고는 [?]로 표시합니다. 출전을 지어내지 말고, 모르면 모른다고 씁니다.

다음 원문을 번역해 주세요.

<원문>
莊周夢爲蝴蝶，蝴蝶又變爲莊周。
莊周之夢爲蝴蝶耶？
蝴蝶之夢爲莊周耶？
終不能辨之！
</원문>

당신은 고전 구절의 전고(典故)와 출전을 찾는 연구 보조자입니다. 각 후보마다 원문 구절, 추정 출전(서명·편명), 근거를 적습니다. 출전이 확실하지 않으면 '불확실'이라고 쓰고, 존재 여부를 확인할 수 없는 문헌은 제시하지 않습니다. 결과는 연구자가 원문 데이터베이스에서 직접 확인할 후보 목록입니다.

다음 구절의 출전 후보를 찾아 주세요.

<구절>
莊周夢爲蝴蝶，蝴蝶又變爲莊周。
莊周之夢爲蝴蝶耶？
蝴蝶之夢爲莊周耶？
終不能辨之！
</구절>


챗봇 없이도 이어서 할 수 있도록, 책에 실은 Claude의 답(2026.11.14.)을 아래에 넣어 두었습니다. 같은 질문에도 답은 매번 조금씩 달라질 수 있습니다. 직접 받은 답이 있으면 이 글을 바꿔 붙여 넣으세요.

In [3]:
answer_translation = """장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변하였다.
장주가 꿈에 나비가 된 것인가?
나비가 꿈에 장주가 된 것인가?
끝내 이를 가려내지 못한다!

[?] 둘째 문장의 ‘莊周之夢爲蝴蝶耶’에서 ‘之’를 ‘장주의 꿈’으로 읽을지, 주어와 서술어 사이의 어조사로 보아 ‘장주가 꿈에’로 읽을지 확신이 없습니다. 위 번역은 뒤의 해석을 따랐습니다."""
answer_sources = """후보 1. 『莊子』 「齊物論」 (신뢰도: 높음)
- 원문 구절: 昔者莊周夢爲胡蝶 … 不知周之夢爲胡蝶與？胡蝶之夢爲周與？周與胡蝶，則必有分矣，此之謂物化。
- 근거: 장주, 꿈, 나비가 서로 바뀌는 구도와 꿈과 현실을 묻는 반복 질문의 형식이 같습니다. 제시된 구절은 인용이라기보다 이 이야기를 바꿔 쓴 것으로 보입니다(‘胡蝶’이 ‘蝴蝶’으로, 끝말 ‘與’가 ‘耶’로 다릅니다).

다른 출전은 확실하게 제시할 수 있는 것이 없어 적지 않습니다. 위 후보도 원문 데이터베이스에서 직접 확인해 주세요."""
print(answer_translation)
print()
print(answer_sources)

장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변하였다.
장주가 꿈에 나비가 된 것인가?
나비가 꿈에 장주가 된 것인가?
끝내 이를 가려내지 못한다!

[?] 둘째 문장의 ‘莊周之夢爲蝴蝶耶’에서 ‘之’를 ‘장주의 꿈’으로 읽을지, 주어와 서술어 사이의 어조사로 보아 ‘장주가 꿈에’로 읽을지 확신이 없습니다. 위 번역은 뒤의 해석을 따랐습니다.

후보 1. 『莊子』 「齊物論」 (신뢰도: 높음)
- 원문 구절: 昔者莊周夢爲胡蝶 … 不知周之夢爲胡蝶與？胡蝶之夢爲周與？周與胡蝶，則必有分矣，此之謂物化。
- 근거: 장주, 꿈, 나비가 서로 바뀌는 구도와 꿈과 현실을 묻는 반복 질문의 형식이 같습니다. 제시된 구절은 인용이라기보다 이 이야기를 바꿔 쓴 것으로 보입니다(‘胡蝶’이 ‘蝴蝶’으로, 끝말 ‘與’가 ‘耶’로 다릅니다).

다른 출전은 확실하게 제시할 수 있는 것이 없어 적지 않습니다. 위 후보도 원문 데이터베이스에서 직접 확인해 주세요.


**꼭 확인하세요:** AI가 내놓은 출전은 후보일 뿐입니다. 그럴듯한 책 이름도 존재하지 않을 수 있습니다. 다음 단계에서 컴퓨터와 원문으로 확인합니다.

## 6.2 컴퓨터와 원문으로 검증하기

첫째, AI의 답에 적힌 한자 구절이 『장자』 「齊物論」 원문에 그대로 있는지 찾습니다. 글자열이 있는지만 알려 주며, 뜻이 맞는지는 사람이 읽어 판단합니다.

In [4]:
from kclit import classics
zhuangzi_text = classics.qiwulun()
llm.check_quotes(answer_sources, zhuangzi_text)

,구절,원문에 있음
0,昔者莊周夢爲胡蝶,True
1,不知周之夢爲胡蝶與,True
2,胡蝶之夢爲周與,True
3,則必有分矣,True
4,此之謂物化,True


둘째, 4장과 같은 방법으로 이 구절이 『장자』 「齊物論」의 어느 문장과 겹치는지 찾고 AI가 말한 출전과 비교하세요.

In [5]:
from kclit import reuse, text
zhuangzi = text.split_sentences(zhuangzi_text)
check = reuse.find_reuse({"구운몽 16회 구절": passage.splitlines()}, {"莊子·齊物論": zhuangzi}, n=2, min_score=0.15, min_shared=4)
reuse.to_frame(check)[["target_line", "source_line", "score", "shared"]]

,target_line,source_line,score,shared
0,蝴蝶之夢爲莊周耶？,胡蝶之夢爲周與？,0.300,蝶之夢為
1,莊周之夢爲蝴蝶耶？,不知周之夢爲胡蝶與？,0.250,周之夢為
2,莊周夢爲蝴蝶，蝴蝶又變爲莊周。,昔者莊周夢爲胡蝶，栩栩然胡蝶也，自喻適志與，不知周也。,0.107,莊周夢為
3,莊周之夢爲蝴蝶耶？,胡蝶之夢爲周與？,0.182,之夢為
4,蝴蝶之夢爲莊周耶？,不知周之夢爲胡蝶與？,0.154,之夢為


AI의 출전 후보가 위 결과와 같은지 비교합니다. 같다면 원문의 문장을 직접 읽어 글자를 대조하고(『장자』는 ‘胡蝶’, 『구운몽』은 ‘蝴蝶’), 다르거나 원문에서 찾을 수 없다면 환각을 의심하고 기록합니다. 컴퓨터가 찾은 것도 ‘읽어 볼 후보’입니다.

## 6.3 원문과 번역 맞추기

원문과 번역문을 문장 단위로 짝짓고, 짝이 없는 문장(생략·첨가)과 유사도가 낮은 짝(의역 의심)을 표시합니다. 아래 번역은 예시용으로 직접 쓴 것이며 일부러 마지막 문장을 뺐습니다.

In [6]:
from kclit import embed, align
src = ["莊周夢爲蝴蝶，蝴蝶又變爲莊周。", "莊周之夢爲蝴蝶耶？", "蝴蝶之夢爲莊周耶？", "終不能辨之！"]
tgt = ["장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변했다.", "장주의 꿈이 나비가 된 것인가?", "나비의 꿈이 장주가 된 것인가?"]
m = embed.sentence_model()
sim = embed.cosine_matrix(embed.encode(m, src), embed.encode(m, tgt))
for s, t, score in align.align(sim):
    print(align.label((s, t, score)), [src[i] for i in s], [tgt[j] for j in t], score)

대응 ['莊周夢爲蝴蝶，蝴蝶又變爲莊周。'] ['장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변했다.'] 0.501
대응 ['莊周之夢爲蝴蝶耶？'] ['장주의 꿈이 나비가 된 것인가?'] 0.651
대응 ['蝴蝶之夢爲莊周耶？'] ['나비의 꿈이 장주가 된 것인가?'] 0.619
생략 ['終不能辨之！'] [] 0.0


In [7]:
# 합치기 기준(merge_penalty)을 낮추면 결과가 어떻게 달라질까?
for s, t, score in align.align(sim, merge_penalty=0.1):
    print(align.label((s, t, score)), [src[i] for i in s], [tgt[j] for j in t], score)

대응 ['莊周夢爲蝴蝶，蝴蝶又變爲莊周。'] ['장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변했다.'] 0.501
대응 ['莊周之夢爲蝴蝶耶？'] ['장주의 꿈이 나비가 된 것인가?'] 0.651
대응 ['蝴蝶之夢爲莊周耶？', '終不能辨之！'] ['나비의 꿈이 장주가 된 것인가?'] 0.539


기준을 낮추면 마지막 문장을 앞 문장과 합쳐서 ‘대응’으로 처리합니다. 어떤 기준으로 돌렸는지 기록해 두세요.

## 6.4 AI를 믿어도 될까

AI를 쓴 작업은 모델, 프롬프트, 날짜, 검증 방법을 기록표에 남깁니다. 논문과 저서에 AI 활용 사실을 밝힐 때 근거가 됩니다.

In [8]:
llm.log_use("ai_log.csv", "번역 초안", "구운몽 16회 장주 구절 번역", "원문 대조 후 직접 교정", model="Claude(대화창)")
llm.log_use("ai_log.csv", "출전 후보", "구운몽 16회 장주 구절 출전", "구절 대조(check_quotes)와 莊子 齊物論 원문 확인", model="Claude(대화창)")
import pandas as pd; pd.read_csv("ai_log.csv")

,날짜,모델,작업,프롬프트,검증 방법,메모
0,2026-11-14,Claude(대화창),번역 초안,구운몽 16회 장주 구절 번역,원문 대조 후 직접 교정,NaN
1,2026-11-14,Claude(대화창),출전 후보,구운몽 16회 장주 구절 출전,구절 대조(check_quotes)와 莊子 齊物論 원문 확인,NaN
